In [29]:
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

In [30]:
from cuttingstock.cleaning import load_data, clean_data, clean_stock
raw_stock_df = load_data('stock.csv')
stock_df = clean_stock(raw_stock_df)
stock_df.head()


Detected semicolon-separated file: stock.csv
Successfully loaded data from stock.csv
Data shape: 1386 rows, 10 columns
Columns: ['โรงงาน  ', '  วันที่ผลิต ', '   ชนิดรายการจ่ายออกสต็อค  ', '  หมายเลขม้วนกระดาษ   ', '   ชนิดกระดาษ    ', '    เครื่องหมายส่ง   ', '     ขนาด (นิ้ว)   ', '     น้ำหนัก (กิโลกรัม)    ', '        ความหนา        ', '      ความยาว']
Sample data:
shape: (5, 10)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ โรงงาน   ┆   วันที่ผลิต  ┆ ชนิดรายการ ┆ หมายเลขม้ว ┆ … ┆ ขนาด (นิ้ว) ┆ น้ำหนัก     ┆ ความหนา   ┆ ความยาว   │
│ ---      ┆ ---       ┆ จ่ายออกสต็อ ┆ นกระดาษ   ┆   ┆ ---       ┆ (กิโลกรัม)  ┆ ---       ┆ ---       │
│ str      ┆ str       ┆ ค         ┆ ---       ┆   ┆ str       ┆ ---       ┆ str       ┆ str       │
│          ┆           ┆ ---       ┆ str       ┆   ┆           ┆ str       ┆           ┆           │
│          ┆           ┆ str       ┆           ┆   ┆           ┆           ┆           ┆      

roll_number,roll_type,roll_size,length
str,str,i64,i64
"""03080731608900""","""K420""",85,6911
"""03080731611300""","""K420""",85,6790
"""03080731611500""","""K420""",85,6662
"""03080831612000""","""K420""",85,6554
"""03080831612300""","""K420""",85,6518


In [31]:
def update_stock_data(stock_df):
    """อัปเดต ROLL_SPECS จาก DataFrame สต็อกที่ทำความสะอาดแล้ว ให้มีโครงสร้างตาม mock-up"""
    new_roll_specs = {}
    for row in stock_df.iter_rows(named=True):
        roll_number = str(row['roll_number']).strip()
        width = str(row['roll_size']).strip()
        material = str(row['roll_type']).strip()
        length = row['length']
        #TEST
        length = 10000000
        if width not in new_roll_specs:
            new_roll_specs[width] = {}
        if material not in new_roll_specs[width]:
            new_roll_specs[width][material] = {}
        # ใช้ key ที่เพิ่มขึ้นเรื่อยๆ สำหรับแต่ละม้วนภายใต้ width/material เดียวกัน
        roll_key = len(new_roll_specs[width][material]) + 1
        new_roll_specs[width][material][roll_key] = {
            'id': roll_number,
            'length': length
            }
    return new_roll_specs
roll_specs = update_stock_data(stock_df)
roll_specs    


{'85': {'K420': {1: {'id': '03080731608900', 'length': 10000000},
   2: {'id': '03080731611300', 'length': 10000000},
   3: {'id': '03080731611500', 'length': 10000000},
   4: {'id': '03080831612000', 'length': 10000000},
   5: {'id': '03080831612300', 'length': 10000000},
   6: {'id': '03080831618600', 'length': 10000000}},
  'KAC185': {1: {'id': '16112338290500', 'length': 10000000},
   2: {'id': '25022256115100', 'length': 10000000},
   3: {'id': '25022256115400', 'length': 10000000}},
  'KM150': {1: {'id': '18090412545500', 'length': 10000000}},
  'KAR225': {1: {'id': '18090732709200', 'length': 10000000}},
  'CM147': {1: {'id': '18122714856600', 'length': 10000000},
   2: {'id': '19080239745700', 'length': 10000000},
   3: {'id': '19080239745900', 'length': 10000000}},
  'KBR160': {1: {'id': '19020435835100', 'length': 10000000}},
  'KS231': {1: {'id': '20052050737900', 'length': 10000000},
   2: {'id': '25062618508500', 'length': 10000000},
   3: {'id': '25062618508800', 'length'

In [32]:
raw_order_df = load_data('order.csv')
order_df = clean_data(raw_order_df, suggestion_mode=True)
order_df.head()


Detected semicolon-separated file: order.csv
Successfully loaded data from order.csv
Data shape: 248 rows, 24 columns
Columns: [' เลขที่ใบสั่งขาย', 'ลำดับที่สั่งส่ง', 'กำหนดส่ง       ', 'จำนวนสั่งส่ง   ', 'สถานะใบสั่งส่ง', 'เกินได้', 'รหัสสินค้า', 'จำนวนสั่งผลิต', 'กว้าง', 'ยาว', 'ผลิตได้', 'ประเภทกล่อง', 'ทับเส้น', 'ซ้าย', 'กลาง', 'ขวา', 'ชั้น', 'กระดาษหน้า', 'ลอนC', 'กระดาษกลาง', 'ลอนB', 'กระดาษหลัง ', ' v.noprod ', 'v.cancel']
Sample data:
shape: (5, 24)
┌────────────┬───────────┬────────────┬────────────┬───┬────────┬───────────┬───────────┬──────────┐
│ เลขที่ใบสั่งขา ┆ ลำดับที่สั่งส่ง ┆ กำหนดส่ง    ┆ จำนวนสั่งส่ง  ┆ … ┆ ลอนB   ┆ กระดาษหลัง ┆ v.noprod  ┆ v.cancel │
│ ย          ┆ ---       ┆ ---        ┆ ---        ┆   ┆ ---    ┆ ---       ┆ ---       ┆ ---      │
│ ---        ┆ str       ┆ str        ┆ str        ┆   ┆ str    ┆ str       ┆ bool      ┆ bool     │
│ str        ┆           ┆            ┆            ┆   ┆        ┆           ┆           ┆          │
╞════════════╪════

due_date,order_number,width,length,demand,quantity,type,component_type,front,c,middle,b,back,die_cut
date,str,f64,f64,i64,i64,str,str,str,str,str,str,str,i64
2025-09-18,"""12181251202-1""",10.5256,40.7678,5000,5100,"""X""","""A""","""KA185""","""""","""""","""CM127""","""KB160""",1
2025-09-18,"""12181251200-1""",10.5256,40.7678,5000,5100,"""X""","""A""","""WLK174""","""""","""""","""CM127""","""KB160""",1
2025-09-18,"""12181251199-1""",10.5256,40.7678,10000,10100,"""X""","""A""","""KA185""","""""","""""","""CM127""","""KB160""",1
2025-09-18,"""12181251191-1""",11.9036,35.8072,2750,2850,"""N""","""E""","""KS231""","""CM127""","""""","""""","""KB160""",1
2025-09-18,"""12181251188-1""",16.5561,57.6318,12760,12860,"""N""","""A""","""KB160""","""CM127""","""CM127""","""CM127""","""KB160""",1


In [33]:
from cuttingstock.core import generate_suggestions
suggestions = generate_suggestions(order_df, roll_specs, '2')
suggestions
bad_suggest = generate_suggestions(order_df, roll_specs, '2', True)
bad_suggest

2025-09-18 08:44:46,375 - cuttingstock.utils - INFO - Suggestions generated successfully | Details: count: 107, factory: 2
2025-09-18 08:44:46,383 - cuttingstock.utils - INFO - Suggestions generated successfully | Details: count: 107, factory: 2


[{'width': '82',
  'spec': {'front': 'KAC125',
   'c': '',
   'middle': '',
   'b': 'CM112',
   'back': 'CME100'}},
 {'width': '82',
  'spec': {'front': 'WLW154',
   'c': '',
   'middle': '',
   'b': 'CM112',
   'back': 'KB160'}},
 {'width': '82',
  'spec': {'front': 'KI128',
   'c': '',
   'middle': '',
   'b': 'CM127',
   'back': 'CM127'}},
 {'width': '82',
  'spec': {'front': 'KS121',
   'c': 'CM127',
   'middle': '',
   'b': '',
   'back': 'CM127'}},
 {'width': '82',
  'spec': {'front': 'KAC185',
   'c': 'CM127',
   'middle': '',
   'b': '',
   'back': 'KB160'}},
 {'width': '82',
  'spec': {'front': 'KB160',
   'c': 'CM127',
   'middle': '',
   'b': '',
   'back': 'KB160'}},
 {'width': '82',
  'spec': {'front': 'KS161',
   'c': 'CM127',
   'middle': '',
   'b': '',
   'back': 'KB160'}},
 {'width': '82',
  'spec': {'front': 'KS161',
   'c': '',
   'middle': '',
   'b': 'CM127',
   'back': 'KB160'}},
 {'width': '82',
  'spec': {'front': 'KS231',
   'c': 'CM127',
   'middle': '',
   '

In [ ]:
import pandas as pd

def count_changes(data, order_df, samewidth=False):
    prev = None
    change_records = []
    processed_order = 0
    for idx, item in enumerate(data):
        current = (
            item['width'],
            item['spec'].get('front', ''),
            item['spec'].get('c', ''),
            item['spec'].get('middle', ''),
            item['spec'].get('b', ''),
            item['spec'].get('back', '')
        )
        change = {'index': idx, 'front': 0, 'c': 0, 'middle': 0, 'b': 0, 'back': 0}
        if prev is not None:
            if current[1] != prev[1]:
                change['front'] = 1
            if current[2] != prev[2]:
                change['c'] = 1
            if current[3] != prev[3]:
                change['middle'] = 1
            if current[4] != prev[4]:
                change['b'] = 1
            if current[5] != prev[5]:
                change['back'] = 1
            if not samewidth:
                if current[0] != prev[0]:
                    for i in ['front', 'c', 'middle', 'b', 'back']:
                        change[i] = 1
        change_records.append(change)
        front = item['spec'].get('front', '')
        b = item['spec'].get('b', '')
        middle = item['spec'].get('middle', '')
        c = item['spec'].get('c', '')
        back = item['spec'].get('back', '')
        processed_order += len(clean_data(order_df, front=front, c=c, middle=middle, b=b, back=back))
        prev = current 
    change_df = pd.DataFrame(change_records)
    print("Change DataFrame:")
    print(change_df)
    print("Number of processed:", processed_order)
    return change_df, processed_order

changes, processed = count_changes(suggestions, raw_order_df, samewidth=True) 
real_changes, real_processed = count_changes(suggestions, raw_order_df) 
badchanges, badprocessed = count_changes(bad_suggest, raw_order_df) 

Starting data cleaning...
วันที่กำหนดส่งขั้นต่ำใน DataFrame (หลังการแยกวิเคราะห์และลบค่าว่าง): 2025-09-18
วันที่กำหนดส่งสูงสุดใน DataFrame (หลังการแยกวิเคราะห์และลบค่าว่าง): 2025-09-18
Data after date filtering:
shape: (5, 14)
┌────────────┬───────────────┬─────────┬─────────┬───┬────────┬────────┬────────┬─────────┐
│ due_date   ┆ order_number  ┆ width   ┆ length  ┆ … ┆ middle ┆ b      ┆ back   ┆ die_cut │
│ ---        ┆ ---           ┆ ---     ┆ ---     ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---     │
│ date       ┆ str           ┆ f64     ┆ f64     ┆   ┆ str    ┆ str    ┆ str    ┆ i64     │
╞════════════╪═══════════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪═════════╡
│ 2025-09-18 ┆ 12181251202-1 ┆ 10.5256 ┆ 40.7678 ┆ … ┆ null   ┆ CM127  ┆ KB160  ┆ 1       │
│ 2025-09-18 ┆ 12181251200-1 ┆ 10.5256 ┆ 40.7678 ┆ … ┆ null   ┆ CM127  ┆ KB160  ┆ 1       │
│ 2025-09-18 ┆ 12181251199-1 ┆ 10.5256 ┆ 40.7678 ┆ … ┆ null   ┆ CM127  ┆ KB160  ┆ 1       │
│ 2025-09-18 ┆ 12181251191-1 ┆ 11.903

In [35]:
print(changes, processed)
print(badchanges, badprocessed)

     index  front  c  middle  b  back
0        0      0  0       0  0     0
1        1      1  1       1  1     1
2        2      1  1       1  1     1
3        3      1  1       1  1     1
4        4      1  1       1  1     1
..     ...    ... ..     ... ..   ...
102    102      1  1       1  1     1
103    103      1  1       1  1     1
104    104      1  0       0  0     1
105    105      1  0       0  0     0
106    106      1  0       0  0     0

[107 rows x 6 columns] 574
     index  front  c  middle  b  back
0        0      0  0       0  0     0
1        1      1  0       0  0     1
2        2      1  0       0  1     1
3        3      1  1       0  1     0
4        4      1  0       0  0     1
..     ...    ... ..     ... ..   ...
102    102      1  0       0  0     1
103    103      1  0       0  0     1
104    104      1  0       0  0     0
105    105      1  0       0  0     0
106    106      0  1       1  1     0

[107 rows x 6 columns] 574


In [ ]:

# Use blue shades for normal, red shades for bad
normal_colors = ['#1f77b4', '#3399cc', '#66b3ff', '#005c99', '#003366']
bad_colors = ['#d62728', '#ff6666', '#ff9999', '#990000', '#660000']

fig = go.Figure()
for idx, col in enumerate(['front', 'c', 'middle', 'b', 'back']):
    fig.add_trace(go.Scatter(
        x=changes.index, y=changes[col].cumsum(),
        mode='lines', name=f'{col.capitalize()}',
        line=dict(color=normal_colors[idx])
    ))
    fig.add_trace(go.Scatter(
        x=badchanges.index, y=badchanges[col].cumsum(),
        mode='lines', name=f'Bad {col.capitalize()}',
        line=dict(color=bad_colors[idx])
    ))
fig.update_layout(title='Changes')
waste = (changes[['front', 'c', 'middle', 'b', 'back']].sum(axis=1) * 0.2).cumsum()
badwaste = (badchanges[['front', 'c', 'middle', 'b', 'back']].sum(axis=1) * 0.2).cumsum()
cols = ['front', 'c', 'middle', 'b', 'back']

def stackbarplot(data, cols, colors, title):
    fig = go.Figure()
    for idx, col in enumerate(cols):
        fig.add_bar(
            x=data.index,
            y=data[col].cumsum(),
            name=col.capitalize(),
            marker_color=colors[idx],
        )
    fig.update_layout(
        barmode='stack',
        title=title
    )
    fig.show()

print(f"Bad waste compare to waste: {((badwaste.iloc[-1] - waste.iloc[-1]) / waste.iloc[-1]) * 100:.2f}%")
fig.show()
stackbarplot(real_changes,cols,normal_colors, title='Sort by Spec (Paul Suggestions)')
stackbarplot(changes,cols,normal_colors, title='Sort by Spec Same Width (Paul Suggestions)')
stackbarplot(badchanges,cols,bad_colors, title='Sort by Width Suggestions')


Bad waste compare to waste: -52.07%
